In [20]:
!pip install pandas

In [93]:
#Importing required Libraries
import pandas as pd
import re
import numpy as np

In [94]:
#Import the dataset cc_calls and billings
cc_calls=pd.read_csv("Dataset/cc_calls.csv",low_memory=False)
billing_df=pd.read_csv("Dataset/billings.csv",low_memory=False)

## Filtering the Billings data

In [95]:
# Removing duplicate Co_Ref entries while keeping the newest renewal date
billing_df['Co_Ref'].value_counts().head()
billing_clean = billing_df.sort_values('Prospect_Renewal_Date').drop_duplicates(
    'Co_Ref', keep='last'
)
len(billing_clean)

47826

In [32]:
billing_clean['Prospect_Renewal_Date'].isna().sum()

np.int64(0)

In [96]:
#Joining the Billings and cc_calls with respect to co_Ref
df_calls_merged = cc_calls.merge(
    billing_clean[['Co_Ref', 'Prospect_Renewal_Date']],
    on='Co_Ref',
    how='inner'
)
len(df_calls_merged)

31208

# Exploratory Data Analysis (EDA)

In [55]:
df_calls_merged.info()
len(df_calls_merged)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31208 entries, 0 to 31207
Data columns (total 34 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Contact_ID                                31208 non-null  float64
 1   Call_Date                                 31208 non-null  object 
 2   Direction                                 31208 non-null  object 
 3   cc_care_package                           31074 non-null  object 
 4   cc_care_package_discussed                 31074 non-null  object 
 5   cc_urgency_getting_on_site                31074 non-null  object 
 6   cc_external_consultant                    31074 non-null  object 
 7   cc_agent_cross_sell_attempt               31074 non-null  object 
 8   cc_customer_issues_concerns               31074 non-null  object 
 9   cc_business_struggles_financial_hardship  31074 non-null  object 
 10  cc_call_initiated_by              

31208

In [56]:
df_calls_merged.describe()

,Contact_ID,Analysed_Call,Call_Year
count,3.120800e+04,31208.0,31208.000000
mean,6.417159e+11,1.0,2024.848981
std,4.884207e+10,0.0,0.412468
min,1.936150e+11,1.0,2024.000000
25%,5.949900e+11,1.0,2025.000000
50%,6.512520e+11,1.0,2025.000000
75%,6.856320e+11,1.0,2025.000000
max,6.918350e+11,1.0,2026.000000


In [67]:
df_calls_merged.isna().sum()

Contact_ID                                  0
Call_Date                                   0
Direction                                   0
cc_care_package                             0
cc_care_package_discussed                   0
cc_urgency_getting_on_site                  0
cc_external_consultant                      0
cc_agent_cross_sell_attempt                 0
cc_customer_issues_concerns                 0
cc_business_struggles_financial_hardship    0
cc_call_initiated_by                        0
cc_questionnaire_completion                 0
cc_chasing_response                         0
cc_issues_within_questionnaire              0
cc_login_issues                             0
cc_platform_issues                          0
cc_dissatisfaction_time_to_complete         0
cc_process_complexity_concerns              0
cc_questions_harder_than_expected           0
cc_dissatisfaction_support                  0
cc_contractor_sentiment                     0
cc_contractor_sentiment_start_scor

In [98]:
# Fix data types for date columns
date_cols = [
    "Call_Date",
    "Prospect_Renewal_Date",
]
for col in date_cols:
    if col in df_calls_merged.columns:
        parsed = pd.to_datetime(
            df_calls_merged[col],
            dayfirst=True,
        )
        df_calls_merged[col] = parsed.combine_first(df_calls_merged[col])

# convert Contact_ID and Co_Ref to string
df_calls_merged['Contact_ID'] = df_calls_merged['Contact_ID'].astype('Int64').astype('string')
df_calls_merged['Co_Ref'] = df_calls_merged['Co_Ref'].astype('string')

# Convert all object-type columns into category type
cat_cols = df_calls_merged.select_dtypes(include='object').columns
df_calls_merged[cat_cols] = df_calls_merged[cat_cols].astype('category')

In [99]:
#Convert score values to numerical values
score_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score'
]

df_calls_merged[score_cols] = df_calls_merged[score_cols].apply(
    pd.to_numeric, errors='coerce'
)

## Replacing Null Values

In [63]:
# 1. Handle categorical columns
for col in cat_cols:
    if "Unknown" not in df_calls_merged[col].cat.categories:
        df_calls_merged[col] = df_calls_merged[col].cat.add_categories(["Unknown"])
    df_calls_merged[col] = df_calls_merged[col].fillna("Unknown")

In [64]:
# 2. Handle numeric score columns
score_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score'
]
df_calls_merged[score_cols] = df_calls_merged[score_cols].fillna(
    df_calls_merged[score_cols].fillna(-1)
)

In [65]:
# drop rows that have Co_Ref=NULL
df_calls_merged['Co_Ref'].isna().sum()
df_calls_merged = df_calls_merged.dropna(subset=['Co_Ref'])

In [74]:
yes_no_cols = [
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact', 
    'cc_refund_discussed', 'cc_contractor_suggest_leave', 
    'cc_contractor_complained'
]
for col in yes_no_cols:
    df_calls_merged[col] = df_calls_merged[col].apply(lambda x: 'No' if str(x).lower().strip() in ['no', '', 'nan', 'none','not'] else 'Yes')
    df_calls_merged[col]= df_calls_merged[col].fillna('Unknown')

## Remove Unnecessary Column

In [80]:
df_calls_merged = df_calls_merged.drop(columns=['cc_contractor_sentiment'])
df_calls_merged = df_calls_merged.drop(columns=['Analysed_Call'])

In [81]:
df_calls_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31208 entries, 0 to 31207
Data columns (total 32 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Contact_ID                                31208 non-null  string        
 1   Call_Date                                 31208 non-null  datetime64[ns]
 2   Direction                                 31208 non-null  category      
 3   cc_care_package                           31208 non-null  category      
 4   cc_care_package_discussed                 31208 non-null  category      
 5   cc_urgency_getting_on_site                31208 non-null  category      
 6   cc_external_consultant                    31208 non-null  category      
 7   cc_agent_cross_sell_attempt               31208 non-null  category      
 8   cc_customer_issues_concerns               31208 non-null  category      
 9   cc_business_struggles_financ

In [101]:
#Do sentiment analysis for the fields that contain descriptive sentence
def clean_messy_binary(val):
    if pd.isna(val) or str(val).strip() == "":
        return "Unknown"
    
    text = str(val).strip().lower()

    # explicit yes/no
    if text in ["yes", "y", "true", "1"]:
        return "Yes"
    if text in ["no", "n", "false", "0", "[yes/no]"]:
        return "No"

    if len(text.split()) > 3:
        return "Yes"

    # if contains helpful/action words
    keywords_yes = [
        "help", "support", "assist", "review", "imply", "process",
        "accreditation", "contractor", "updated", "investigated"
    ]
    if any(k in text for k in keywords_yes):
        return "Yes"
    #If it is digits convert that to "Yes" 
    if re.fullmatch(r"\d+", text):
        return "Yes"

    return "Unknown"


In [102]:
#Converting the messy columns to yes/no 
messy_cols = [
    "cc_agent_cross_sell_attempt",
    "cc_customer_issues_concerns",
    "cc_business_struggles_financial_hardship",
    "cc_chasing_response",
    "cc_issues_within_questionnaire",
    "cc_login_issues",
    "cc_platform_issues",
    "cc_process_complexity_concerns",
    "cc_questions_harder_than_expected",
    "cc_dissatisfaction_support",
    ]

for col in messy_cols:
    df_calls_merged[col] = df_calls_merged[col].apply(clean_messy_binary).astype("category")

In [103]:
for col in messy_cols:
    print(col, df_calls_merged[col].value_counts(dropna=False))

cc_agent_cross_sell_attempt cc_agent_cross_sell_attempt
No         29920
Yes         1153
NaN          134
Unknown        1
Name: count, dtype: int64
cc_customer_issues_concerns cc_customer_issues_concerns
No         27922
Yes         3138
NaN          134
Unknown       14
Name: count, dtype: int64
cc_business_struggles_financial_hardship cc_business_struggles_financial_hardship
No         30144
Yes          924
NaN          134
Unknown        6
Name: count, dtype: int64
cc_chasing_response cc_chasing_response
No         23611
Yes         7561
NaN           29
Unknown        7
Name: count, dtype: int64
cc_issues_within_questionnaire cc_issues_within_questionnaire
No         25904
Yes         4138
Unknown      733
NaN          433
Name: count, dtype: int64
cc_login_issues cc_login_issues
No     29625
Yes     1553
NaN       30
Name: count, dtype: int64
cc_platform_issues cc_platform_issues
No     29066
Yes     2112
NaN       30
Name: count, dtype: int64
cc_process_complexity_concerns cc_

In [104]:
df_calls_merged.to_csv('cc_calls_cleaned.csv', index=False)

# Applying Filter

In [105]:
df_calls_merged = df_calls_merged[
    df_calls_merged['Call_Date'] > df_calls_merged['Prospect_Renewal_Date']
]

In [106]:
df_calls_merged.to_csv('cc_calls_cleaned_filtered.csv', index=False)